# MNIST MLP — Kaggle Notebook training

**Before running this notebook**, in the Kaggle UI:
1. Go to your published dataset's page (`kaggle.com/datasets/<you>/mnist-mlp-pipeline`) and use its **⋮ menu → New notebook** — this attaches the dataset automatically. (Searching for a private dataset from inside an existing notebook's "Add Input" can be unreliable / laggy to index.)
2. In that notebook, **Add Input** → search `digit-recognizer` → add the card with the trophy icon, marked "Ongoing" / "Getting Started" (not the similarly-named community datasets below it).
3. **Settings → Accelerator → GPU T4 x2** (or P100). Requires phone-verifying your Kaggle account once, if you haven't already.
4. **Settings → Environment Preferences → Internet → On** (needed for the `pip install` cell below).

**Note on input paths**: this Kaggle environment nests inputs rather than using the flat `/kaggle/input/<slug>/` layout some older docs describe:
- Datasets mount at `/kaggle/input/datasets/<owner>/<dataset-slug>/`
- Competitions mount at `/kaggle/input/competitions/<competition-slug>/`

The cells below already use these paths. Kaggle also auto-extracts the uploaded `src.zip`/`configs.zip` into real folders — no unzip step needed, just add the dataset path to `sys.path`.

Re-run `scripts/publish_kaggle_dataset.py` locally and re-add/update the input whenever `src/` or `configs/` changes — Kaggle datasets don't auto-sync with local edits.

In [ ]:
!pip install -q pytorch-lightning torchmetrics optuna python-dotenv

In [ ]:
import sys

DATASET_DIR = '/kaggle/input/datasets/ivannamor/mnist-mlp-pipeline'
sys.path.insert(0, DATASET_DIR)
print('project code ready')

In [ ]:
from src.config import ExperimentConfig

config = ExperimentConfig.from_yaml(f'{DATASET_DIR}/configs/baseline.yaml')
config.data.data_dir = '/kaggle/input/competitions/digit-recognizer'
config.trainer.experiment_name = 'baseline'
config

## Baseline training run (GPU)

In [ ]:
from src.train import train

trainer = train(config)

## Hyperparameter search (Optuna)

Reuses `search.objective`, pointed at the same Kaggle data path. Start with a small `n_trials` to gauge how long each trial takes within the GPU quota before committing to a larger search. No checkpoints are saved per trial (kept lightweight) — only the winning hyperparameters come out of this step; the next cell retrains that configuration for real.

In [ ]:
import optuna
from src.search import objective

study = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner())
study.optimize(lambda trial: objective(trial, config), n_trials=20)

print('Best value:', study.best_value)
print('Best params:', study.best_params)

## Retrain the winning configuration

The search above only reports the best hyperparameters, not a saved model. This retrains that exact configuration once, with checkpointing on, to get the model that the final evaluation runs against.

In [ ]:
from src.search import build_config

best_config = build_config(study.best_params, config)
best_config.trainer.experiment_name = 'best_from_search'
best_trainer = train(best_config)

## Final test-set evaluation

Run this once, after the retrain above — not per trial. This is the number to report.

In [ ]:
from pathlib import Path
from src.evaluate import evaluate

checkpoint_path = best_trainer.checkpoint_callback.best_model_path
evaluate(best_config, checkpoint_path, Path('/kaggle/working/evaluation'))